# Step 21: Amazon Chronos-2 Zero-Shot Foundation Model Experiment

This notebook evaluates **Amazon Chronos-2** (`amazon/chronos-2`) as a zero-shot, in-context time series forecasting foundation model for APMC mandi price prediction across 4 crops:
1. **Rice**
2. **Tomato**
3. **Wheat**
4. **Cotton**

### Objectives & Guarantees
- **Horizon**: Primary forecast horizon is 3 observations ahead ($t+3$), aligning with the validated project target `price_after_3_observations`.
- **Zero Data Leakage**: At any forecast cutoff date $T$, the context series only contains observations $\le T$.
- **Baselines**: Compared directly against the **Persistence Baseline** ($y_{t+3} = y_t$), **7-Observation Moving Average (MA-7)**, and the previous best **Tabular ML Models**.
- **Decision Rule**: Chronos-2 becomes the production forecasting method **only** if it genuinely outperforms persistence on the untouched held-out test sets.

In [1]:
import sys
from pathlib import Path
import pandas as pd
BASE_DIR = Path('..').resolve()
if str(BASE_DIR) not in sys.path: sys.path.insert(0, str(BASE_DIR))

from src.chronos_forecast import get_chronos_pipeline
pipeline = get_chronos_pipeline(device='cpu')
print('Chronos-2 Pipeline Ready!')


/Users/moksh/Desktop/MandiMitra-ML/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading amazon/chronos-2 on device=cpu...


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 170/170 [00:00<00:00, 19183.50it/s]

Chronos-2 Pipeline Ready!


## 1. Run Chronological Evaluation Across All 4 Crops

In [2]:
from src.chronos_evaluation import run_full_chronos_experiment
summary_df = run_full_chronos_experiment()
display(summary_df)


MANDIMITRA — CHRONOS-2 BENCHMARK EXPERIMENT

Evaluating RICE: 3 markets, 336 test rows


Collected 327 evaluation instances for rice. Running Chronos-2 inference...


Completed 327 evaluations for rice in 0.7s | Chronos MAE: 126.68 vs Persistence: 120.92

Evaluating TOMATO: 4 markets, 423 test rows


Collected 411 evaluation instances for tomato. Running Chronos-2 inference...


Completed 411 evaluations for tomato in 0.8s | Chronos MAE: 405.51 vs Persistence: 409.59

Evaluating WHEAT: 21 markets, 2430 test rows


Collected 1188 evaluation instances for wheat. Running Chronos-2 inference...


Completed 1188 evaluations for wheat in 2.3s | Chronos MAE: 62.75 vs Persistence: 67.55

Evaluating COTTON: 1 markets, 83 test rows
Collected 80 evaluation instances for cotton. Running Chronos-2 inference...


Completed 80 evaluations for cotton in 0.2s | Chronos MAE: 135.18 vs Persistence: 103.62

CHRONOS-2 BENCHMARK RESULTS
  crop  n_eval_samples  chronos_mae  chronos_rmse  chronos_mape  chronos_r2  persistence_mae  persistence_rmse  persistence_mape  ma7_mae existing_ml_model  existing_ml_mae  improvement_vs_persistence_pct best_method
  Rice             327     126.6772      315.4252        2.7512      0.8250         120.9205          298.5066            2.6131 134.1782 Gradient Boosting         232.1300                           -4.76 persistence
Tomato             411     405.5090      545.9339       16.1421      0.6738         409.5864          604.7440           16.0279 420.6465     Random Forest         434.3620                            1.00 persistence
 Wheat            1188      62.7540      109.1155        2.0965      0.9487          67.5530          127.6874            2.2773  71.2136 Gradient Boosting          68.2390                            7.10   chronos-2
Cotton        

,crop,n_eval_samples,chronos_mae,chronos_rmse,chronos_mape,chronos_r2,persistence_mae,persistence_rmse,persistence_mape,ma7_mae,existing_ml_model,existing_ml_mae,improvement_vs_persistence_pct,best_method
0,Rice,327,126.6772,315.4252,2.7512,0.8250,120.9205,298.5066,2.6131,134.1782,Gradient Boosting,232.1300,-4.76,persistence
1,Tomato,411,405.5090,545.9339,16.1421,0.6738,409.5864,604.7440,16.0279,420.6465,Random Forest,434.3620,1.00,persistence
2,Wheat,1188,62.7540,109.1155,2.0965,0.9487,67.5530,127.6874,2.2773,71.2136,Gradient Boosting,68.2390,7.10,chronos-2
3,Cotton,80,135.1770,169.4615,1.6615,0.3602,103.6250,164.4023,1.2717,122.8036,Gradient Boosting,212.2251,-30.45,persistence


## 2. Model Performance Comparison vs Baselines

In [3]:
comp_df = pd.read_csv(BASE_DIR / 'outputs' / 'final' / 'chronos2_model_comparison.csv')
display(comp_df)


,crop,n_eval_samples,chronos_mae,chronos_rmse,chronos_mape,chronos_r2,persistence_mae,persistence_rmse,persistence_mape,ma7_mae,existing_ml_model,existing_ml_mae,improvement_vs_persistence_pct,best_method
0,Rice,327,126.6772,315.4252,2.7512,0.8250,120.9205,298.5066,2.6131,134.1782,Gradient Boosting,232.1300,-4.76,persistence
1,Tomato,411,405.5090,545.9339,16.1421,0.6738,409.5864,604.7440,16.0279,420.6465,Random Forest,434.3620,1.00,persistence
2,Wheat,1188,62.7540,109.1155,2.0965,0.9487,67.5530,127.6874,2.2773,71.2136,Gradient Boosting,68.2390,7.10,chronos-2
3,Cotton,80,135.1770,169.4615,1.6615,0.3602,103.6250,164.4023,1.2717,122.8036,Gradient Boosting,212.2251,-30.45,persistence


## 3. Market-Level Generalization Performance

In [4]:
mkt_df = pd.read_csv(BASE_DIR / 'outputs' / 'final' / 'chronos2_market_performance.csv')
display(mkt_df.head(20))


,crop,market,n_samples,chronos_mae,persistence_mae,improvement_pct
0,Rice,APMC Alibagh,111,64.43,44.59,-44.49
1,Rice,APMC Murud,111,64.43,44.59,-44.49
2,Rice,APMC Palghar,105,258.28,282.30,8.51
3,Tomato,APMC Kamthi,104,364.57,345.96,-5.38
4,Tomato,APMC Panvel,106,494.49,504.34,1.95
5,Tomato,Pune(Manjri),107,487.27,496.26,1.81
6,Tomato,Pune(Pimpri),94,257.40,274.47,6.22
7,Wheat,APMC Akola,47,43.13,44.77,3.65
8,Wheat,APMC Amarawati,43,21.78,35.26,38.23
9,Wheat,APMC Chattrapati Sambhajinagar,45,79.31,84.73,6.40


## 4. Probabilistic Uncertainty & Prediction Intervals

In [5]:
unc_df = pd.read_csv(BASE_DIR / 'outputs' / 'final' / 'chronos2_uncertainty_results.csv')
display(unc_df)


,crop,nominal_interval,empirical_coverage_pct,average_interval_width_rs,average_actual_price,width_as_pct_of_price
0,Rice,80% (q0.1 - q0.9),90.83,448.77,4126.36,10.88
1,Tomato,80% (q0.1 - q0.9),78.59,1250.67,2663.65,46.95
2,Wheat,80% (q0.1 - q0.9),77.86,187.86,3000.21,6.26
3,Cotton,80% (q0.1 - q0.9),65.00,322.08,8072.31,3.99


## 5. Conclusions and Production Method Selection

Analysis of whether the foundation model improves upon persistence or tabular models, followed by justified selection.